In [1]:
import torch 
import numpy as np 
import os
from pathlib import Path
import importlib
import src.util_tfrecord as util_tfrecords
import os
import tensorflow as tf
import yaml
from pytorch_lightning import Trainer

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')

In [2]:
config_path = "config/binaural_attn/word_task_mixed_cue_v04.yml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['hparas']['batch_size'] = 64
config['audio']['rep_kwargs']['rep_on_gpu'] = True

path_to_tf_records = Path("/om/scratch/Sat/msaddler/dataset_multitask/v04")



# dev lightning module 

In [ ]:

from collections import namedtuple
from typing import List, Tuple, Optional, Union
import numpy as np
import torch
from torchmetrics.classification import Accuracy
from pytorch_lightning import LightningModule

import src.audio_transforms as at
import src.custom_modules as cm
from src.base_word_loc_cnn import AuditoryCNN

## TO DO:  Import new dataset class


# def get_memory_usage():
#     mem = psutil.virtual_memory()
#     return mem.used / 1024 ** 3


class AttnBiasConstraint(object):
    def __init__(self, min_val=0, max_val=1):
        self.min = min_val
        self.max = max_val
        
    def __call__(self, module):
        if hasattr(module,'bias'):
            b = module.bias.data
            module.bias.data = b.clamp(self.min, self.max)

class AttnSlopeConstraint(object):
    def __init__(self, min_val=0):
        self.min = min_val

    def __call__(self, module):
        if hasattr(module,'slope'):
            s = module.slope.data
            module.slope.data = s.clamp(self.min) # no max -> max = inf   

class BinauralAttentionModule(LightningModule):
    def __init__(
        self,
        config: dict,
    ):
        super().__init__()
        # Make config sent to init
        self.config = config
        self.audio_config = config['audio']
        self.corpora_config = config['corpus']
        self.model_config = config['model']
        self.hparas_config = config['hparas']

        self.corpora_name = config.get('corpora_name', False)

        num_words = config['model']['num_classes']['num_words']
        num_locs = config['model']['num_classes']['num_locs']

        self.audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            # at.BinauralCombineWithRandomDBSNR(low_snr=config['noise_kwargs']['low_snr'], high_snr=config['noise_kwargs']['high_snr']),
            at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
        ])

        self.test_step = self._test_step

        if self.run_timit:
            self.test_step = self.test_timit 
            self.audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.UnsqueezeAudio(dim=0),
            ])

        # Init Model
        # fc_attn_only = self.model_config.get('fc_attn_only', False) 
        # fc_size = self.model_config.get('fc_size', 4096)
        # global_avg_cue = self.model_config.get('global_avg_cue', False)
        # Get model architecture
        model = AuditoryCNN(num_words=800, num_locs=512) 
        # check if torch version 2 or greater - if so, compile model
        self.model = torch.compile(model, mode="reduce-overhead")

        # Add input rep to model or audio transforms
        self.rep_on_gpu = self.audio_config['rep_kwargs']['rep_on_gpu']
        self.coch_gram = cm.AttnAudioInputRepresentation(**self.audio_config)


        # Losses
        self.loss_fn = torch.nn.CrossEntropyLoss()

        self.train_acc = torch.nn.ModuleDict({'word':Accuracy(task="multiclass", num_classes=num_words), 
                                                'location':Accuracy(task="multiclass", num_classes=num_locs)})
        self.valid_acc = torch.nn.ModuleDict({'word':Accuracy(task="multiclass", num_classes=num_words),
                                                'location':Accuracy(task="multiclass", num_classes=num_locs)})
        self.test_acc = torch.nn.ModuleDict({'word':Accuracy(task="multiclass", num_classes=num_words),
                                                'location':Accuracy(task="multiclass", num_classes=num_locs)})
        self.test_confusion = torch.nn.ModuleDict({'word': Accuracy(task="multiclass", num_classes=num_words),
                                                 'location': Accuracy(task="multiclass", num_classes=num_locs)})
        self.accuracy = {'train': self.train_acc,
                         'val': self.valid_acc,
                         'test': self.test_acc,
                         'test_confusion': self.test_confusion
                        }

        # Constraints
        # self.attn_modules = [mod for name, mod in self.model._modules.items() if 'attn' in name]
        # self.bias_constraint = AttnBiasConstraint(min_val=0, max_val=1)
        # self.constrain_slope = self.model_config['attn_constraints'].get('slope', False)
        # if self.constrain_slope:
        #     self.slope_constraint = AttnSlopeConstraint(min_val=0)

        # Optimizer
        opt = getattr(torch.optim, self.hparas_config['optimizer'])
        model_params = [{'params': self.model.parameters()}]
        self.optimizer = opt(model_params, lr=self.hparas_config['lr'], eps=self.hparas_config['eps'])       
        
    def _step(self, batch, batch_idx, step_type):
        if batch is None:
            return None
        # cue_features, cue_mask_ixs, scene_features, labels = batch

        cue_features, scene_features = self.coch_gram(cue_features, scene_features)

        # self() is self.forward()
        outputs = self(cue_features, scene_features, cue_mask_ixs)
        
        word, location = outputs
        word_label = labels[:,0]
        location_label = labels[:,1]
        word_loss = self.loss_fn(word, word_label)
        loc_loss = self.loss_fn(location, location_label)
        loss = word_loss + loc_loss
        self.accuracy[step_type]['word'](word, word_label) # word accuracy
        self.accuracy[step_type]['location'](location, location_label) # location accuracy
        self.log(f"{step_type}_loss", loss.detach(), on_step=True, on_epoch=False, prog_bar=True)
        self.log(f"{step_type}_word_acc", self.accuracy[step_type]['word'], on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{step_type}_location_acc", self.accuracy[step_type]['location'], on_step=False, on_epoch=True, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return [self.optimizer]

    def forward(self, sound: torch.tensor):
        outputs = self.model(sound)
        # Outputs here are logits
        return outputs

    def training_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "val")

    def train_dataloader(self):
        filenames = [path.as_posix() for path in config["data"]["root"].glob("train/*.tfrecords")]
        dataset = util_tfrecords.get_dataset_from_tfrecords(
                filenames,
                feature_description='config_feature_description.pkl',
                features_to_exclude=['list_'],
                bytes_description='config_bytes_description.pkl',
                compression_type='GZIP',
                eval_mode=False,
                buffer_size_prefetch=tf.data.AUTOTUNE,
                buffer_size_shuffle=100,
                batch_size=64,
                shuffle_seed=None,
                densify_downsample_factors=None,
                densify_jitter_indices=None,
                densify_dtype=tf.float32,
                filter_function=None,
                map_function=None,
                verbose=False)
        return dataset

    def val_dataloader(self):
        filenames = [path.as_posix() for path in config["data"]["root"].glob("valid/*.tfrecords")]
        dataset = util_tfrecords.get_dataset_from_tfrecords(
                filenames,
                feature_description='config_feature_description.pkl',
                features_to_exclude=['list_'],
                bytes_description='config_bytes_description.pkl',
                compression_type='GZIP',
                eval_mode=True,
                buffer_size_prefetch=tf.data.AUTOTUNE,
                buffer_size_shuffle=100,
                batch_size=64,
                shuffle_seed=None,
                densify_downsample_factors=None,
                densify_jitter_indices=None,
                densify_dtype=tf.float32,
                filter_function=None,
                map_function=None,
                verbose=False)
        return dataset


/om/scratch/Mon/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v04
mixture_percentages={'voice_and_location': 0.33, 'voice_only': 0.33, 'location_only': 0.33}
1006 files in train concat dataset


In [ ]:

trainer = Trainer(
    precision="16-mixed",
    limit_val_batches=0.0,
    num_nodes=1,
    devices=2, # was gpus=1,
    # detect_anomaly=True,
    strategy="ddp_notebook",
    accelerator="gpu",
)
trainer.fit(module)

/om2/user/imgriff/conda_envs/torch_2_cuda_12/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:191: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3.10 /om2/user/imgriff/conda_envs/torch_2_cuda_12/lib ...
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/om2/user/imgriff/conda_envs/torch_2_cuda_12/lib/python3.10/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
/om2/user/imgriff/conda_envs/torch_2_cuda_12/lib/python3.10/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs

/om/scratch/Mon/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v04
/om/scratch/Mon/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v04
mixture_percentages={'voice_and_location': 0.33, 'voice_only': 0.33, 'location_only': 0.33}
mixture_percentages={'voice_and_location': 0.33, 'voice_only': 0.33, 'location_only': 0.33}1006 files in train concat dataset

1006 files in train concat dataset
len training set = 46276len training set = 46276



/om2/user/imgriff/conda_envs/torch_2_cuda_12/lib/python3.10/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
